# Optuna hyperparameter optimization

### Import libraries and set configs

In [ ]:
import ast
import json
import pandas as pd

import optuna

from cfg.config import SHARED


class CFG:
    n_trials = 250
    # maximum number of simultaneously opened trades for backtest metric
    max_num_simult_trades = SHARED.max_num_simult_trades
    # significance level, that is used to conduct a t-test between 2 models
    optimize_alpha = 0.2
    n_repeats = 1
    n_folds = 8
    min_precision = SHARED.min_precision
    TP = SHARED.TP
    SL = SHARED.SL
    slippage = SHARED.slippage
    test_time_days = 90

### Load the train data

In [ ]:
train_df = pd.read_pickle("data/train_df.pkl")

# all data for the last 90 days are test
test_date = train_df["time"].max() - pd.to_timedelta(CFG.test_time_days, unit="D")

fi = pd.read_csv("model/features/feature_importance.csv")

### Optimize

In [ ]:
from utils.optimization_utils import make_objective

with open("model/bybit_tickers.json", "r") as f:
    bybit_tickers = json.load(f)

df_optuna_more_info = pd.DataFrame(columns=["result", "backtest_result", "oof_conf_score",
                                            "profit_objects", "oof_conf_obj_num", "scores"])
df_optuna_more_info.to_csv("model/optuna/optuna_lgbm_info.csv", index=False)

objective = make_objective(
    train_df, 
    test_date, 
    fi, 
    bybit_tickers, 
    TP=CFG.TP - CFG.slippage,
    SL=CFG.SL + CFG.slippage,
    n_folds=CFG.n_folds, 
    optimize_alpha=CFG.optimize_alpha, 
    min_precision=CFG.min_precision,
)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=CFG.n_trials)

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

df_optuna = study.trials_dataframe()
df_optuna = df_optuna.sort_values("value", ascending=False)
df_optuna.to_csv("optuna/optuna_lgbm.csv", index=False)

display(df_optuna.head(10))

/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[I 2026-06-10 20:08:10,361] A new study created in memory with name: no-name-204b12b6-bdf6-4529-87ef-7dd88261ab63
/home/alex/Repos/sigbot/ml/utils/optimization_utils.py:209: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_optuna_more_info = pd.concat([df_optuna_more_info, tmp])
[I 2026-06-10 20:15:43,847] Trial 0 finished with value: 73.99051364365971 and parameters: {'boosting_type': 'gbdt', 'n_estim

avg conf score 163.5860229276896 is better than best score 147.8765782493369, but p-value 0.8816288999325373 is more than alpha 0.2


[I 2026-06-10 20:59:27,716] Trial 11 finished with value: 160.00962717593944 and parameters: {'boosting_type': 'goss', 'n_estimators': 1239, 'learning_rate': 0.27628197841890234, 'reg_alpha': 1.7937703807014674e-06, 'reg_lambda': 5.752817377833548e-06, 'max_depth': 7, 'num_leaves': 315, 'colsample_bytree': 0.47472803929534285, 'max_bin': 157, 'is_unbalance': True, 'high_bound': 0.3826245018958675, 'low_bound': 0.0993294323155806, 'feature_num': 371, 'corr_thresh': 0.9765253771601472, 'sample_weight': 'cos', 'class_weight': None}. Best is trial 11 with value: 160.00962717593944.


avg conf score 167.30580051369864 is better than best score 149.73612249736246, but p-value 0.4152707483298926 is more than alpha 0.2


[I 2026-06-10 21:01:54,691] Trial 12 finished with value: 102.4521135371179 and parameters: {'boosting_type': 'goss', 'n_estimators': 1321, 'learning_rate': 0.2433226405501711, 'reg_alpha': 6.737306994302883e-06, 'reg_lambda': 1.020793159798185e-05, 'max_depth': 7, 'num_leaves': 298, 'colsample_bytree': 0.4981742752676316, 'max_bin': 144, 'is_unbalance': True, 'high_bound': 0.3870165320956367, 'low_bound': 0.09892689895472333, 'feature_num': 369, 'corr_thresh': 0.9855535002181147, 'sample_weight': 'cos', 'class_weight': None}. Best is trial 11 with value: 160.00962717593944.
[I 2026-06-10 21:04:13,927] Trial 13 finished with value: 133.7306220047923 and parameters: {'boosting_type': 'goss', 'n_estimators': 1292, 'learning_rate': 0.1307885959796222, 'reg_alpha': 4.083709670696427e-06, 'reg_lambda': 8.870740667129925e-06, 'max_depth': 7, 'num_leaves': 241, 'colsample_bytree': 0.4809188003167479, 'max_bin': 140, 'is_unbalance': True, 'high_bound': 0.3843187434582529, 'low_bound': 0.099765

...

[I 2026-06-11 03:19:45,826] Trial 114 finished with value: 0.0 and parameters: {'boosting_type': 'dart', 'n_estimators': 2472, 'learning_rate': 0.00019915968571928628, 'reg_alpha': 0.6805897894844322, 'reg_lambda': 0.0004814328510844733, 'max_depth': 5, 'num_leaves': 166, 'colsample_bytree': 0.7212409850309938, 'max_bin': 170, 'is_unbalance': False, 'high_bound': 0.5629201312674925, 'low_bound': 0.07943964329580855, 'feature_num': 69, 'corr_thresh': 0.9274002170555065, 'sample_weight': 'linear', 'subsample': 0.699419014529249}. Best is trial 113 with value: 214.04668011484526.
[I 2026-06-11 03:25:50,868] Trial 115 finished with value: 119.26683760683763 and parameters: {'boosting_type': 'dart', 'n_estimators': 2306, 'learning_rate': 0.0021759317083932064, 'reg_alpha': 0.2346169320366106, 'reg_lambda': 0.00023882696103043663, 'max_depth': 5, 'num_leaves': 156, 'colsample_bytree': 0.7607682596458792, 'max_bin': 154, 'is_unbalance': False, 'high_bound': 0.5847299584658504, 'low_bound': 0.

avg conf score 386.3071274416063 is better than best score 377.6490051850959, but p-value 0.7346944549251584 is more than alpha 0.2


[I 2026-06-11 14:49:31,270] Trial 220 finished with value: 206.26168998476243 and parameters: {'boosting_type': 'dart', 'n_estimators': 2414, 'learning_rate': 0.25506535316269474, 'reg_alpha': 0.1434329875885783, 'reg_lambda': 0.03445238858950834, 'max_depth': 4, 'num_leaves': 252, 'colsample_bytree': 0.891691709554469, 'max_bin': 165, 'is_unbalance': True, 'high_bound': 0.5360225806633755, 'low_bound': 0.03675207775535799, 'feature_num': 52, 'corr_thresh': 0.9706289365165209, 'sample_weight': None, 'subsample': 0.7370017993448191, 'class_weight': 'balanced'}. Best is trial 219 with value: 379.94605302968404.
[I 2026-06-11 14:55:48,715] Trial 221 finished with value: 196.4197341901563 and parameters: {'boosting_type': 'dart', 'n_estimators': 2488, 'learning_rate': 0.1946507699501485, 'reg_alpha': 0.49855235934496145, 'reg_lambda': 0.02811141241598076, 'max_depth': 4, 'num_leaves': 260, 'colsample_bytree': 0.8741848848410072, 'max_bin': 172, 'is_unbalance': True, 'high_bound': 0.5244167

Number of finished trials: 250
Best trial:
  Value: 379.94605302968404
  Params: 
    boosting_type: dart
    n_estimators: 2445
    learning_rate: 0.1969438727735549
    reg_alpha: 0.15619467637911458
    reg_lambda: 0.023633971701923735
    max_depth: 4
    num_leaves: 255
    colsample_bytree: 0.887251061084062
    max_bin: 169
    is_unbalance: True
    high_bound: 0.5253338458643013
    low_bound: 0.041634186151336953
    feature_num: 63
    corr_thresh: 0.9581562610688437
    sample_weight: None
    subsample: 0.7463830288952731
    class_weight: balanced


,number,value,datetime_start,datetime_complete,duration,params_boosting_type,params_class_weight,params_colsample_bytree,params_corr_thresh,params_feature_num,...,params_low_bound,params_max_bin,params_max_depth,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_sample_weight,params_subsample,state
219,219,379.946053,2026-06-11 14:37:27.061154,2026-06-11 14:43:33.674130,0 days 00:06:06.612976,dart,balanced,0.887251,0.958156,63,...,0.041634,169,4,2445,255,0.156195,0.023634,None,0.746383,COMPLETE
124,124,377.649005,2026-06-11 04:11:25.669535,2026-06-11 04:17:31.882351,0 days 00:06:06.212816,dart,NaN,0.874894,0.910133,67,...,0.041746,191,4,2573,223,0.040826,0.000555,linear,0.742151,COMPLETE
198,198,373.810449,2026-06-11 12:18:48.187734,2026-06-11 12:25:12.091282,0 days 00:06:23.903548,dart,balanced,0.839500,0.966111,70,...,0.046242,167,4,2518,248,0.803296,0.000827,None,0.773103,COMPLETE
129,129,369.087500,2026-06-11 04:42:42.737595,2026-06-11 04:49:00.750016,0 days 00:06:18.012421,dart,balanced,0.865950,0.972662,67,...,0.047099,189,4,2580,253,0.041690,0.001826,linear,0.740301,COMPLETE
242,242,361.002676,2026-06-11 17:04:00.428140,2026-06-11 17:10:20.896688,0 days 00:06:20.468548,dart,balanced,0.553370,0.919354,73,...,0.038820,189,4,2564,267,0.137058,0.001296,None,0.785363,COMPLETE
118,118,358.717091,2026-06-11 03:39:28.414855,2026-06-11 03:45:40.659810,0 days 00:06:12.244955,dart,NaN,0.706071,0.910585,64,...,0.047405,180,4,2633,210,0.021798,0.000819,linear,0.722773,COMPLETE
217,217,356.216252,2026-06-11 14:25:06.966759,2026-06-11 14:31:27.099594,0 days 00:06:20.132835,dart,balanced,0.859714,0.953243,61,...,0.040847,186,4,2554,260,0.172331,0.009033,None,0.711030,COMPLETE
156,156,354.995582,2026-06-11 07:43:49.707791,2026-06-11 07:50:29.200165,0 days 00:06:39.492374,dart,balanced,0.844492,0.947092,81,...,0.052988,197,4,2690,265,0.031565,0.001763,linear,0.644794,COMPLETE
226,226,353.880582,2026-06-11 15:21:00.559805,2026-06-11 15:27:06.415448,0 days 00:06:05.855643,dart,balanced,0.877028,0.947444,59,...,0.046929,189,4,2467,271,0.160353,0.008139,None,0.762041,COMPLETE
119,119,352.545783,2026-06-11 03:45:40.660406,2026-06-11 03:51:50.862640,0 days 00:06:10.202234,dart,NaN,0.703382,0.908999,61,...,0.045895,204,4,2625,147,0.072159,0.000720,linear,0.719360,COMPLETE
